# ⚖️ LawGPT — RAG Pipeline
**Legal Research Assistant powered by LangChain + Pinecone + OpenAI**

### Textbooks ingested
| File | Subject | Pages |
|------|---------|-------|
| `1.pdf` | Torts: Theory and Practice (4th ed.) | 20 |
| `2.pdf` | Copyright Law: Cases and Materials (v7.0) | 729 |
| `3.pdf` | Professional Responsibility (2nd ed.) | 1100 |

### Pipeline overview
```
PDFs → PyPDFLoader → RecursiveCharacterTextSplitter
     → HuggingFace Embeddings (all-MiniLM-L6-v2)
     → Pinecone Vector Store (index: lawgpt)
     → Retrieval Chain (GPT-4o + legal system prompt)
```


## 0 · Setup

In [1]:
import os
os.chdir("../")   # Move to project root
%pwd

'c:\\Users\\Asus\\OneDrive\\Desktop'

## 1 · Imports

In [2]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
from typing import List
import os

## 2 · Load PDF Textbooks

All three law textbooks live in `data/` (same folder convention as HealthMate-AI).
We load them individually so we can tag each chunk with the correct source title —
useful later for citations in answers.


In [3]:
# ── Map filenames → human-readable titles for metadata ──────────────────
LAW_BOOKS = {
    "1.pdf": "Torts: Theory and Practice (4th ed.)",
    "2.pdf": "Copyright Law: Cases and Materials (v7.0)",
    "3.pdf": "Professional Responsibility (2nd ed.)",
}

DATA_DIR = "C:/Users/Asus/OneDrive/Desktop/NEW/data"   # Put all three PDFs here


def load_law_pdfs(data_dir: str, book_map: dict) -> List[Document]:
    """
    Load each law textbook PDF individually so we can attach a clean
    'title' field to every chunk's metadata alongside 'source'.
    """
    all_docs: List[Document] = []

    for filename, title in book_map.items():
        filepath = os.path.join(data_dir, filename)
        if not os.path.exists(filepath):
            print(f"[SKIP] Not found: {filepath}")
            continue

        loader = PyPDFLoader(filepath)
        docs   = loader.load()

        # Enrich metadata
        for doc in docs:
            doc.metadata["title"]  = title
            doc.metadata["source"] = filepath

        all_docs.extend(docs)
        print(f"[OK]   {title}  →  {len(docs)} pages loaded")

    print(f"\nTotal pages loaded: {len(all_docs)}")
    return all_docs


raw_docs = load_law_pdfs(DATA_DIR, LAW_BOOKS)

[OK]   Torts: Theory and Practice (4th ed.)  →  20 pages loaded
[OK]   Copyright Law: Cases and Materials (v7.0)  →  729 pages loaded
[OK]   Professional Responsibility (2nd ed.)  →  1100 pages loaded

Total pages loaded: 1849


In [4]:
# Spot-check — first 3 docs
raw_docs[:3]

[Document(metadata={'producer': '', 'creator': 'XPP', 'creationdate': '2014-02-28T13:39:13+00:00', 'subject': '', 'author': '', 'keywords': '', 'moddate': '2016-06-06T12:03:37-04:00', 'title': 'Torts: Theory and Practice (4th ed.)', 'source': 'C:/Users/Asus/OneDrive/Desktop/NEW/data\\1.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}, page_content='TORTS:\nTHEORY AND PRACTICE\nFOURTH EDITION'),
 Document(metadata={'producer': '', 'creator': 'XPP', 'creationdate': '2014-02-28T13:39:13+00:00', 'subject': '', 'author': '', 'keywords': '', 'moddate': '2016-06-06T12:03:37-04:00', 'title': 'Torts: Theory and Practice (4th ed.)', 'source': 'C:/Users/Asus/OneDrive/Desktop/NEW/data\\1.pdf', 'total_pages': 20, 'page': 1, 'page_label': '2'}, page_content='LexisNexis Law School Publishing\nAdvisory Board\nPaul Caron\nProfessor of Law\nPepperdine University School of Law\nHerzog Summer Visiting Professor in Taxation\nUniversity of San Diego School of Law\nBridgette Carr\nClinical Professor of

## 3 · Filter & Clean Documents

Keep only `page_content`, `source`, and `title` in metadata.
This mirrors the `filter_to_minimal_docs` helper from HealthMate-AI
but adds the `title` field so answers can cite the correct textbook.


In [5]:
def filter_law_docs(docs: List[Document]) -> List[Document]:
    """
    Retain only the fields we actually need.
    Drops empty pages (common in textbook PDFs — blank pages, half-title pages, etc.).
    """
    cleaned: List[Document] = []
    for doc in docs:
        content = doc.page_content.strip()
        if len(content) < 50:          # skip near-empty pages
            continue
        cleaned.append(
            Document(
                page_content=content,
                metadata={
                    "source": doc.metadata.get("source", ""),
                    "title":  doc.metadata.get("title",  "Unknown"),
                    "page":   doc.metadata.get("page",   ""),
                }
            )
        )
    print(f"Pages after filtering: {len(cleaned)}  (removed {len(docs) - len(cleaned)} empty pages)")
    return cleaned


clean_docs = filter_law_docs(raw_docs)

Pages after filtering: 1828  (removed 21 empty pages)


In [6]:
clean_docs[:3]

[Document(metadata={'source': 'C:/Users/Asus/OneDrive/Desktop/NEW/data\\1.pdf', 'title': 'Torts: Theory and Practice (4th ed.)', 'page': 1}, page_content='LexisNexis Law School Publishing\nAdvisory Board\nPaul Caron\nProfessor of Law\nPepperdine University School of Law\nHerzog Summer Visiting Professor in Taxation\nUniversity of San Diego School of Law\nBridgette Carr\nClinical Professor of Law\nUniversity of Michigan Law School\nOlympia Duhart\nProfessor of Law and Director of Lawyering Skills & Values Program\nNova Southeastern University, Shepard Broad Law School\nSamuel Estreicher\nDwight D. Opperman Professor of Law\nDirector, Center for Labor and Employment Law\nNYU School of Law\nSteven I. Friedland\nProfessor of Law and Senior Scholar\nElon University School of Law\nCarole Goldberg\nJonathan D. Varat Distinguished Professor of Law\nUCLA School of Law\nOliver Goodenough\nProfessor of Law\nVermont Law School\nPaul Marcus\nHaynes Professor of Law\nWilliam and Mary Law School\nJoh

## 4 · Text Splitting

Legal text is dense and reference-heavy, so we use a **larger chunk size (800)**
than the HealthMate-AI pipeline (500) to keep legal reasoning intact across sentences.
Overlap is bumped to 100 tokens so a holding or rule that spans a page break
doesn't get split across two chunks without context.


In [7]:
def text_split(docs: List[Document], chunk_size: int = 800, chunk_overlap: int = 100) -> List[Document]:
    """
    Split documents into overlapping chunks suitable for legal RAG retrieval.
    chunk_size=800  → larger than medical (500) to preserve legal reasoning
    chunk_overlap=100 → avoids cutting mid-sentence on holdings / rules
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],   # prefer paragraph breaks
    )
    chunks = splitter.split_documents(docs)
    return chunks


texts_chunk = text_split(clean_docs)
print(f"Total chunks: {len(texts_chunk)}")

Total chunks: 8618


In [8]:
# Distribution by source book
from collections import Counter
title_counts = Counter(c.metadata["title"] for c in texts_chunk)
for title, n in title_counts.items():
    print(f"  {n:>5} chunks  ←  {title}")

     90 chunks  ←  Torts: Theory and Practice (4th ed.)
   4023 chunks  ←  Copyright Law: Cases and Materials (v7.0)
   4505 chunks  ←  Professional Responsibility (2nd ed.)


In [9]:
texts_chunk[:3]

[Document(metadata={'source': 'C:/Users/Asus/OneDrive/Desktop/NEW/data\\1.pdf', 'title': 'Torts: Theory and Practice (4th ed.)', 'page': 1}, page_content='LexisNexis Law School Publishing\nAdvisory Board\nPaul Caron\nProfessor of Law\nPepperdine University School of Law\nHerzog Summer Visiting Professor in Taxation\nUniversity of San Diego School of Law\nBridgette Carr\nClinical Professor of Law\nUniversity of Michigan Law School\nOlympia Duhart\nProfessor of Law and Director of Lawyering Skills & Values Program\nNova Southeastern University, Shepard Broad Law School\nSamuel Estreicher\nDwight D. Opperman Professor of Law\nDirector, Center for Labor and Employment Law\nNYU School of Law\nSteven I. Friedland\nProfessor of Law and Senior Scholar\nElon University School of Law\nCarole Goldberg\nJonathan D. Varat Distinguished Professor of Law\nUCLA School of Law\nOliver Goodenough\nProfessor of Law\nVermont Law School\nPaul Marcus\nHaynes Professor of Law'),
 Document(metadata={'source': 

## 5 · Embeddings

Same model as HealthMate-AI (`all-MiniLM-L6-v2`, 384-dim).
Works well for legal text — fast, cheap, and strong semantic similarity.
No change needed here.


In [10]:
def download_embeddings(model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> HuggingFaceEmbeddings:
    """Download and return HuggingFace sentence-transformer embeddings."""
    return HuggingFaceEmbeddings(model_name=model_name)


embedding = download_embeddings()
embedding

C:\Users\Asus\AppData\Local\Temp\ipykernel_15828\3881716392.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [11]:
# Sanity check — embed a legal query
vec = embedding.embed_query("What is the standard of care in negligence?")
print(f"Vector dimension: {len(vec)}")
print(f"First 10 values:  {vec[:10]}")

Vector dimension: 384
First 10 values:  [0.002149693202227354, 0.0840902328491211, -0.044604960829019547, -0.028962591663002968, -0.05965611711144447, 0.006417630240321159, -0.04792388528585434, 0.11346977204084396, -0.05126504600048065, 0.03207281604409218]


## 6 · Pinecone Vector Store

New index: **`lawgpt`**.
Same spec: 384-dim cosine similarity on AWS us-east-1.


In [19]:
load_dotenv(dotenv_path=r"C:\Users\Asus\OneDrive\Desktop\NEW\.env")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"]   = OPENAI_API_KEY

pc = Pinecone(api_key=PINECONE_API_KEY)
pc

In [20]:
INDEX_NAME = "lawgpt"

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name      = INDEX_NAME,
        dimension = 384,            # all-MiniLM-L6-v2 output dim
        metric    = "cosine",
        spec      = ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"[CREATED] Index '{INDEX_NAME}'")
else:
    print(f"[EXISTS]  Index '{INDEX_NAME}' already present")

index = pc.Index(INDEX_NAME)
index.describe_index_stats()

[CREATED] Index 'lawgpt'


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

## 7 · Upsert Chunks into Pinecone

Run the cell below **only on first ingest** (or when you add new books).
Comment it out afterwards and use `from_existing_index` to avoid re-embedding.


In [21]:
# ── FIRST TIME ONLY — comment out after initial ingest ──────────────────
# Batched upsert to avoid Pinecone request-size limits on large corpora

from langchain_pinecone import PineconeVectorStore

BATCH_SIZE = 100

print(f"Upserting {len(texts_chunk)} chunks in batches of {BATCH_SIZE}...")

for i in range(0, len(texts_chunk), BATCH_SIZE):
    batch = texts_chunk[i : i + BATCH_SIZE]
    if i == 0:
        docsearch = PineconeVectorStore.from_documents(
            documents  = batch,
            embedding  = embedding,
            index_name = INDEX_NAME,
        )
    else:
        docsearch.add_documents(batch)
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"  ... upserted {min(i + BATCH_SIZE, len(texts_chunk))} / {len(texts_chunk)}")

print("Upsert complete.")

Upserting 8618 chunks in batches of 100...
  ... upserted 100 / 8618
  ... upserted 1100 / 8618
  ... upserted 2100 / 8618
  ... upserted 3100 / 8618
  ... upserted 4100 / 8618
  ... upserted 5100 / 8618
  ... upserted 6100 / 8618
  ... upserted 7100 / 8618
  ... upserted 8100 / 8618
Upsert complete.


In [22]:
# ── AFTER FIRST INGEST — use this cell to reconnect ─────────────────────
docsearch = PineconeVectorStore.from_existing_index(
    index_name = INDEX_NAME,
    embedding  = embedding,
)

## 8 · Retriever

`k=5` instead of 3 — legal questions often need multiple cases or sections
to construct a complete answer.


In [23]:
retriever = docsearch.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 5},
)

In [24]:
# Test retrieval
test_query = "What are the elements of negligence?"
retrieved   = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved)} chunks for: '{test_query}'\n")
for i, doc in enumerate(retrieved, 1):
    title = doc.metadata.get("title", "?")
    page  = doc.metadata.get("page",  "?")
    print(f"--- Chunk {i}  [{title}, p.{page}] ---")
    print(doc.page_content[:300])
    print()

Retrieved 5 chunks for: 'What are the elements of negligence?'

--- Chunk 1  [Torts: Theory and Practice (4th ed.), p.4.0] ---
Preface
This edition brings the text thoroughly up to date in statutory, judicial, and
Restatement developments, and includes law and economic analyses and commentary at
key spots throughout.
The structure of the book builds upon the realities that the most common tort litigation
is in negligence ac

--- Chunk 2  [Torts: Theory and Practice (4th ed.), p.13.0] ---
Notes and Questions . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 590
§ 11.04 SPECIAL CONSIDERATIONS IN COMPARATIVE NEGLIGENCE . . 592
[A] Liability of Multiple Defendants . . . . . . . . . . . . . . . . . . . . . . . . . . . . 592
[1] Joint and Several Liability . . . . . . 

--- Chunk 3  [Copyright Law: Cases and Materials (v7.0), p.580.0] ---
(2) that material or activity was removed or disabled by mistake or misidentification, shall be liable for any

--- Chunk 4  [Torts: Th

## 9 · LLM + Legal System Prompt

The system prompt instructs the model to:
- ground every answer in the retrieved context (the three textbooks)
- cite the source textbook and relevant case names
- flag jurisdictional variations
- add a brief disclaimer when answers could be mistaken for legal advice


In [25]:
chat_model = ChatOpenAI(model="gpt-4o", temperature=0)

In [26]:
LEGAL_SYSTEM_PROMPT = (
    "You are LawGPT, an expert AI legal research assistant. "
    "You have been given excerpts from the following law school textbooks as context:\n"
    "  • Torts: Theory and Practice (4th ed.) — Little, Lidsky & Lande\n"
    "  • Copyright Law: Cases and Materials (v7.0) — Fromer & Sprigman\n"
    "  • Professional Responsibility: A Contemporary Approach (2nd ed.) — Capra & Green\n\n"
    "Rules:\n"
    "1. Base your answer STRICTLY on the provided context. Do not fabricate cases or statutes.\n"
    "2. Cite the source textbook and any case names mentioned in the context.\n"
    "3. Use proper legal terminology (elements, holdings, dicta, majority/minority rule, etc.).\n"
    "4. When a rule varies by jurisdiction, say so explicitly.\n"
    "5. If the context does not contain enough information, say: "
    "   'The provided materials do not cover this topic in sufficient detail.'\n"
    "6. End every substantive answer with: "
    "   '⚠️ This is general legal information, not legal advice. Consult a licensed attorney for your specific situation.'\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", LEGAL_SYSTEM_PROMPT),
    ("human",  "{input}"),
])

In [27]:
question_answer_chain = create_stuff_documents_chain(chat_model, prompt)
rag_chain             = create_retrieval_chain(retriever, question_answer_chain)

## 10 · Example Queries

One example per textbook to verify end-to-end retrieval + generation.


### Torts

In [28]:
response = rag_chain.invoke({"input": "What are the four elements required to establish a negligence claim?"})
print(response["answer"])

The provided materials do not cover this topic in sufficient detail. ⚠️ This is general legal information, not legal advice. Consult a licensed attorney for your specific situation.


In [29]:
response = rag_chain.invoke({"input": "Explain the holding and significance of Palsgraf v. Long Island Railroad."})
print(response["answer"])

The case of Palsgraf v. Long Island Railroad Co. is a seminal case in tort law, particularly in the area of negligence and the concept of duty of care. The holding of the case, as articulated by Chief Judge Benjamin Cardozo, was that the defendant, Long Island Railroad Co., was not liable for the injuries suffered by the plaintiff, Helen Palsgraf, because the harm was not a foreseeable result of the railroad's conduct.

In Palsgraf, the plaintiff was injured when a set of scales fell on her after a package containing fireworks was dropped by a passenger being helped onto a train by railroad employees. The fireworks exploded, causing the scales to fall. The court held that the railroad employees could not have reasonably foreseen that their actions would result in harm to Palsgraf, who was standing some distance away.

The significance of Palsgraf lies in its establishment of the principle that a defendant owes a duty of care only to those plaintiffs who are within the "zone of foreseea

In [30]:
response = rag_chain.invoke({"input": "What is res ipsa loquitur and when does it apply?"})
print(response["answer"])

The provided materials do not cover this topic in sufficient detail. ⚠️ This is general legal information, not legal advice. Consult a licensed attorney for your specific situation.


### Copyright Law

In [31]:
response = rag_chain.invoke({"input": "What are the four fair use factors under copyright law?"})
print(response["answer"])

The four fair use factors under copyright law, as outlined in the provided materials, are:

1. **The purpose and character of the use**: This factor considers whether the use is of a commercial nature or is for nonprofit educational purposes. It focuses on what the copier’s use of the original work accomplishes.

2. **The nature of the copyrighted work**: This factor examines the type of work being used, considering aspects such as whether the work is factual or creative.

3. **The amount and substantiality of the portion used in relation to the copyrighted work as a whole**: This factor assesses how much of the original work is used and whether the portion used is significant or central to the original work.

4. **The effect of the use upon the potential market for or value of the copyrighted work**: This factor evaluates whether the use of the work could serve as a substitute for the original, potentially affecting its market value or depriving the rights holder of significant revenu

In [32]:
response = rag_chain.invoke({"input": "What is the idea-expression dichotomy in copyright law?"})
print(response["answer"])

The idea-expression dichotomy in copyright law is a fundamental principle that distinguishes between ideas and their expression. According to § 102(b) of the copyright statute, copyright protection does not extend to ideas, procedures, processes, systems, methods of operation, concepts, principles, or discoveries, regardless of how they are described, explained, illustrated, or embodied in a work. This means that while the expression of an idea can be protected by copyright, the idea itself cannot be.

The rationale behind this dichotomy is to prevent the monopolization of ideas, which would stifle creativity and innovation. If ideas were protected, it would limit others from building upon those ideas to create new works. The dichotomy ensures that only the specific way an idea is expressed is protected, allowing others to use the underlying idea freely.

In practice, this distinction can be challenging to apply, as seen in cases like Whelan, where the court's approach to distinguishin

### Professional Responsibility

In [33]:
response = rag_chain.invoke({"input": "What are a lawyer's duties of confidentiality under the Model Rules?"})
print(response["answer"])

Under the ABA Model Rules of Professional Conduct, specifically Rule 1.6, a lawyer has a duty not to reveal "information relating to the representation of a client" unless certain conditions are met. This duty of confidentiality is broad and covers information received from both the client and third parties. A lawyer may only disclose such information if the client gives informed consent, the disclosure is impliedly authorized to carry out the representation, or the disclosure is permitted by specific exceptions outlined in the rules.

The duty of confidentiality is broader than the attorney-client privilege because it encompasses a wider range of information, including that which comes from third parties. However, it is also narrower in some respects, as it is subject to certain exceptions that allow for disclosure under specific circumstances.

It is important to note that many states have adopted versions of Rule 1.6 that vary in significant ways from the ABA Model Rule. The ABA pro

In [34]:
response = rag_chain.invoke({"input": "When is a lawyer permitted to withdraw from representation?"})
print(response["answer"])

A lawyer is permitted to withdraw from representation under several circumstances as outlined in Rule 1.16(b) of the rules governing professional responsibility. These circumstances include:

1. Withdrawal can be accomplished without material adverse effect on the interests of the client.
2. The client persists in a course of action involving the lawyer's services that the lawyer reasonably believes is criminal or fraudulent.
3.


## 11 · Source Citation Helper

Utility to print the retrieved source chunks alongside any answer —
useful for debugging retrieval quality and for building a citation panel in the UI.


In [35]:
def ask_with_sources(question: str, chain, ret) -> None:
    """Print answer + the supporting chunks with book/page metadata."""
    response = chain.invoke({"input": question})
    
    print("=" * 70)
    print(f"Q: {question}")
    print("=" * 70)
    print(response["answer"])
    print()
    print("── Supporting Sources ──────────────────────────────────────────────")
    seen = set()
    for doc in response.get("context", []):
        title = doc.metadata.get("title", "Unknown")
        page  = doc.metadata.get("page",  "?")
        key   = (title, page)
        if key in seen:
            continue
        seen.add(key)
        print(f"  [{title}, p.{page}]")
        print(f"  {doc.page_content[:200].strip()}...")
        print()


ask_with_sources(
    "What is the difference between contributory and comparative negligence?",
    rag_chain, retriever
)

Q: What is the difference between contributory and comparative negligence?
The provided materials do not cover this topic in sufficient detail. ⚠️ This is general legal information, not legal advice. Consult a licensed attorney for your specific situation.

── Supporting Sources ──────────────────────────────────────────────
  [Copyright Law: Cases and Materials (v7.0), p.560.0]
  Liability for contributory infringement is based on the nexus between the defendant and the third 
party’s directly infringing activity. Courts deciding a contributory -copyright-infringement claim 
a...

  [Copyright Law: Cases and Materials (v7.0), p.554.0]
  1. Contributory Copyright Infringement 
[7] Contributory copyright infringement is a form of secondary liability with roots in the tort -law concepts of 
enterprise liability and imputed intent…. We h...

  [Copyright Law: Cases and Materials (v7.0), p.548.0]
  [15] Plaintiffs have stated a claim for vicarious copyright infringement.  
Contributory Cop

## 12 · Evaluation

Adapts the `eval_metrices` evaluation from HealthMate-AI.
Replace `data/law_questions.csv` with your own question set.

Expected CSV columns: `question`, `ground_truth_answer`


In [36]:
import pandas as pd
from rouge_score import rouge_scorer

def evaluate_rag_pipeline(
    csv_path : str,
    chain,
    retriever,
    encoding : str = "utf-8",
    max_rows  : int = None,
) -> pd.DataFrame:
    """
    Run the RAG chain over a CSV of questions, compute ROUGE-L against
    ground-truth answers, and return a results DataFrame.
    """
    df = pd.read_csv(csv_path, encoding=encoding)
    if max_rows:
        df = df.head(max_rows)

    scorer  = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    results = []

    for _, row in df.iterrows():
        q   = str(row["question"])
        ref = str(row.get("ground_truth_answer", ""))

        response = chain.invoke({"input": q})
        answer   = response["answer"]

        sources  = list({d.metadata.get("title", "") for d in response.get("context", [])})
        rL       = scorer.score(ref, answer)["rougeL"].fmeasure if ref else None

        results.append({
            "question":        q,
            "generated_answer": answer,
            "ground_truth":    ref,
            "rouge_l":         round(rL, 4) if rL is not None else None,
            "sources":         " | ".join(sources),
        })
        print(f"  [done] {q[:60]}...")

    results_df = pd.DataFrame(results)
    results_df.to_csv("data/lawgpt_eval_results.csv", index=False)
    print(f"\nSaved → data/lawgpt_eval_results.csv")
    print(f"Mean ROUGE-L: {results_df['rouge_l'].mean():.4f}")
    return results_df


# Uncomment to run evaluation:
# results_df = evaluate_rag_pipeline(
#     csv_path  = "data/law_questions.csv",
#     chain     = rag_chain,
#     retriever = retriever,
# )
# results_df.head(10)

## 13 · Adding More Textbooks Later

To ingest a new PDF (e.g., a Contracts casebook):

```python
new_title = "Contracts: Cases and Commentary (3rd ed.)"
new_path  = "data/contracts_casebook.pdf"

new_loader = PyPDFLoader(new_path)
new_raw    = new_loader.load()
for d in new_raw:
    d.metadata["title"]  = new_title
    d.metadata["source"] = new_path

new_clean  = filter_law_docs(new_raw)
new_chunks = text_split(new_clean)

docsearch.add_documents(new_chunks)
print(f"Added {len(new_chunks)} chunks from '{new_title}'")
```

No re-indexing needed — Pinecone upserts are incremental.
